In [25]:
from ollama import Client
from ollama._types import Options
import os
import pandas as pd
from pynvml import *
import time

In [2]:
nvmlInit()
handle = nvmlDeviceGetHandleByIndex(0)

# GPU model name
name = nvmlDeviceGetName(handle)
name = name.replace("NVIDIA GeForce ","")

# Memory info
mem_info = nvmlDeviceGetMemoryInfo(handle)
total_vram_gb = mem_info.total / (1024**3)

# GPU model identifier in the perf tables
gpu_model = f"{name} {total_vram_gb:.0f}GB"

print(f"Testing performance on the following GPU: {gpu_model}")

Testing performance on the following GPU: RTX 5090 32GB


In [3]:
def get_vram_used():
    info = nvmlDeviceGetMemoryInfo(handle)
    return info.used / 10**9

In [4]:
initial_vram = get_vram_used()
print(f"Initial VRAM used: {initial_vram:.3f}GB")

Initial VRAM used: 28.865GB


### Ollama config

"OLLAMA_FLASH_ATTENTION":    {"OLLAMA_FLASH_ATTENTION", FlashAttention(false), "Enabled flash attention"},

    How can I enable Flash Attention?
    Flash Attention is a feature of most modern models that can significantly reduce memory usage as the context size grows. To enable Flash Attention, set the OLLAMA_FLASH_ATTENTION environment variable to 1 when starting the Ollama server.

	// FlashAttention enables the experimental flash attention feature.
	FlashAttention = BoolWithDefault("OLLAMA_FLASH_ATTENTION")

    "OLLAMA_FLASH_ATTENTION":    {"OLLAMA_FLASH_ATTENTION", FlashAttention(false), "Enabled flash attention"},

    To enable Flash Attention, set the `OLLAMA_FLASH_ATTENTION` environment variable to `1`

    // FlashAttention checks if the model should enable flash attention
    func (f GGML) FlashAttention() bool {
    	return slices.Contains([]string{
    		"bert",
    		"gemma3",
    		"gemma4",
    		"glm4moelite",
    		"glmocr",
    		"gptoss", "gpt-oss",
    		"lfm2",
    		"lfm2moe",
    		"mistral3",
    		"nemotron_h", "nemotron_h_moe",
    		"olmo3",
    		"qwen3", "qwen3moe",
    		"qwen35", "qwen35moe",
    		"qwen3next",
    		"qwen3vl", "qwen3vlmoe",
    	}, f.KV().String("general.architecture"))
    }

"OLLAMA_KV_CACHE_TYPE":      {"OLLAMA_KV_CACHE_TYPE", KvCacheType(), "Quantization type for the K/V cache (default: f16)"},

    How can I set the quantization type for the K/V cache?
    The K/V context cache can be quantized to significantly reduce memory usage when Flash Attention is enabled.
    
    To use quantized K/V cache with Ollama you can set the following environment variable:
    
    OLLAMA_KV_CACHE_TYPE - The quantization type for the K/V cache. Default is f16.
    Currently this is a global option - meaning all models will run with the specified quantization type.
    The currently available K/V cache quantization types are:
    
    f16 - high precision and memory usage (default).
    q8_0 - 8-bit quantization, uses approximately 1/2 the memory of f16 with a very small loss in precision, this usually has no noticeable impact on the model's quality (recommended if not using f16).
    q4_0 - 4-bit quantization, uses approximately 1/4 the memory of f16 with a small-medium loss in precision that may be more noticeable at higher context sizes.
    How much the cache quantization impacts the model's response quality will depend on the model and the task. Models that have a high GQA count (e.g. Qwen2) may see a larger impact on precision from quantization than models with a low GQA count.
    
    You may need to experiment with different quantization types to find the best balance between memory usage and quality.

	// KvCacheType is the quantization type for the K/V cache.
	KvCacheType = String("OLLAMA_KV_CACHE_TYPE")

    func kvCacheBytesPerElement(cacheType string) float64 {
    	switch cacheType {
    	case "q8_0":
    		return 1 // 1/2 of fp16
    	case "q4_0":
    		return 0.5 // 1/4 of fp16
    	case "f32":
    		return 4 // f32 (default for recurrent)
    	default:
    		return 2 // f16 (default)
    	}

"OLLAMA_GPU_OVERHEAD":       {"OLLAMA_GPU_OVERHEAD", GpuOverhead(), "Reserve a portion of VRAM per GPU (bytes)"},

	var GpuOverhead = Uint64("OLLAMA_GPU_OVERHEAD", 0)

    available := gpu.FreeMemory - envconfig.GpuOverhead() - gpu.MinimumMemory()

    // MinimumMemory reports the amount of memory that should be set aside
    // on the device for overhead (e.g. VRAM consumed by context structures independent
    // of model allocations)
    func (d DeviceInfo) MinimumMemory() uint64 {
    	if d.Library == "Metal" {
    		return 512 * format.MebiByte
    	}
    	return 457 * format.MebiByte
    }

OLLAMA_NUM_PARALLEL":       {"OLLAMA_NUM_PARALLEL", NumParallel(), "Maximum number of parallel requests"},

	// NumParallel sets the number of parallel model requests. NumParallel can be configured via the OLLAMA_NUM_PARALLEL environment variable.
	NumParallel = Uint("OLLAMA_NUM_PARALLEL", 1)

"OLLAMA_MULTIUSER_CACHE":    {"OLLAMA_MULTIUSER_CACHE", MultiUserCache(), "Optimize prompt caching for multi-user scenarios"},

	// MultiUserCache optimizes prompt caching for multi-user scenarios
	MultiUserCache = Bool("OLLAMA_MULTIUSER_CACHE")

    Multi-user cache enables:
    Cache segmentation
    Different prompt prefixes stored independently
    Cache lookup / matching
    Reuse partial prefixes across requests
    Memory defragmentation
    Reorganizing KV cache blocks dynamically
    Concurrency safety
    Multiple requests accessing cache simultaneously

"OLLAMA_NEW_ENGINE":         {"OLLAMA_NEW_ENGINE", NewEngine(), "Enable the new Ollama engine"},

	// Enable the new Ollama engine
	NewEngine = Bool("OLLAMA_NEW_ENGINE")

    These two flags are closely related:
    
    OLLAMA_MULTIUSER_CACHE
    → controls cache strategy
    OLLAMA_NEW_ENGINE
    → controls entire inference backend
    
    👉 In practice:
    
    Multi-user cache is much more effective with the new engine
    New engine likely implements the advanced caching system

"OLLAMA_CONTEXT_LENGTH":     {"OLLAMA_CONTEXT_LENGTH", ContextLength(), "Context length to use unless otherwise specified (default: 4k/32k/256k based on VRAM)"},

	// ContextLength sets the default context length
	ContextLength = Uint("OLLAMA_CONTEXT_LENGTH", 0)
	
Undocumented

	// DebugLogRequests logs inference requests to disk for replay/debugging.
	DebugLogRequests = Bool("OLLAMA_DEBUG_LOG_REQUESTS")

> OLLAMA_HOST=0.0.0.0 OLLAMA_KEEP_ALIVE=-1 OLLAMA_FLASH_ATTENTION=1 ollama serve &

> OLLAMA_HOST=0.0.0.0 OLLAMA_KEEP_ALIVE=-1 OLLAMA_FLASH_ATTENTION=1 OLLAMA_NEW_ENGINE=1 OLLAMA_MULTIUSER_CACHE=1 ollama serve &

> OLLAMA_HOST=0.0.0.0 OLLAMA_KEEP_ALIVE=-1 OLLAMA_FLASH_ATTENTION=1 OLLAMA_NEW_ENGINE=1 OLLAMA_MULTIUSER_CACHE=1 OLLAMA_KV_CACHE_TYPE=q8_0 ollama serve & 

In [5]:
client = Client()

In [80]:
ollama_version=!ollama --version
ollama_version=ollama_version[0].split()[-1]
ollama_version

'0.20.4-rc1'

In [53]:
def load_model():
    initial_vram = get_vram_used()
    response = client.chat(model=model, messages=[{'role': 'user', 'content': 'say hello'}], options=Options(num_ctx=context_length, num_predict=1))
    return get_vram_used()-initial_vram, response.load_duration/10**9

In [50]:
def compute_prompt_tokens(prompt):
    response = client.chat(model=model, messages=[{'role': 'user', 'content': prompt}], options=Options(num_ctx=context_length, num_predict=100))
    return response.prompt_eval_count

In [63]:
def measure_perfs(prompt_repeat):
    response = client.chat(model=model, messages=[{'role': 'user', 'content': (base_text*prompt_repeat)}], options=Options(num_ctx=context_length, num_predict=100))
    return response.prompt_eval_count,response.prompt_eval_count/(response.prompt_eval_duration/10**9), response.eval_count,response.eval_count/(response.eval_duration/10**9)

In [52]:
def unload_model():
    client.chat(model=model, messages=[{'role': 'user', 'content': 'say goodbye'}], options=Options(num_ctx=context_length, num_predict=1),
    keep_alive=0)
    time.sleep(1)
    return get_vram_used()-initial_vram

In [11]:
context_sizes = [1024*p for p in range(1,8)] + [8192*p for p in range(1,17)] + [16384*p for p in range(9,17)] + [32768*p for p in range(9,17)] + [65536*p for p in range(9,17)]

In [12]:
ctx1k = context_sizes[0]
ctx2k = context_sizes[1]
ctx4k = context_sizes[3]
ctx8k = context_sizes[7]
ctx16k = context_sizes[8]
ctx32k = context_sizes[10]
ctx64k = context_sizes[14]
ctx128k = context_sizes[22]
ctx192k = context_sizes[26]
ctx256k = context_sizes[30]
ctx384k = context_sizes[34]
ctx512k = context_sizes[38]
ctx1M = context_sizes[46]

In [13]:
print(",".join([f"{int(context_sizes[ctx_index]/1024)}k" for ctx_index in range(7,35)]))
print(",".join([f"{context_sizes[ctx_index]}" for ctx_index in range(7,35)]))

8k,16k,24k,32k,40k,48k,56k,64k,72k,80k,88k,96k,104k,112k,120k,128k,144k,160k,176k,192k,208k,224k,240k,256k,288k,320k,352k,384k
8192,16384,24576,32768,40960,49152,57344,65536,73728,81920,90112,98304,106496,114688,122880,131072,147456,163840,180224,196608,212992,229376,245760,262144,294912,327680,360448,393216


In [14]:
base_text = '\nAbsolutely! Let’s unpack this carefully and thoroughly. The concept of a **token** comes up most often in natural language processing (NLP), programming, and blockchain/crypto, but I’ll focus first on the context most relevant to AI models like me, and then touch briefly on other uses for clarity.\n\n---\n\n## **1. Tokens in AI and NLP (Language Models)**\n\nIn the context of language models like GPT:\n\n**A token is essentially a piece of text that the model treats as a unit for processing.**\n\nThink of it as a “chunk” of text, which could be:\n\n* A word (“apple” → 1 token)\n* Part of a word (“unbelievable” → might be split into “un”, “believ”, “able” → 3 tokens)\n* A single character or punctuation mark ("," or "!" → 1 token)\n* Even whitespace in some systems.\n\nLanguage models **do not process text character by character or word by word** in the strict sense; they process these tokens. This is because using tokens allows the model to handle text efficiently while capturing meaningful patterns.\n\n### **Why not just words?**\n\nWords are tricky because:\n\n* Some words are very long and rare.\n* Some languages (like Chinese, Japanese) don’t use spaces to separate words.\n* Using subword tokens balances between vocabulary size and representation power.\n\nBy using tokens, a model can efficiently represent language while keeping the total vocabulary manageable.\n\n---\n\n### **Tokenization: How text becomes tokens**\n\nThe process of breaking text into tokens is called **tokenization**. There are different tokenization methods:\n\n1. **Whitespace tokenization**\n\n   * Splits text at spaces.\n   * `"I love pizza!"` → `["I", "love", "pizza!"]`\n   * Simple, but can miss nuances like punctuation.\n\n2. **Subword tokenization (most modern models use this)**\n\n   * Breaks rare words into smaller pieces.\n   * `"unbelievable"` → `["un", "believ", "able"]`\n   * Helps the model understand new words it hasn\'t seen during training.\n\n3. **Character-level tokenization**\n\n   * Treats each character as a token.\n   * `"cat"` → `["c", "a", "t"]`\n   * Rarely used alone because sequences become very long.\n\n4. **Byte Pair Encoding (BPE)**\n\n   * A popular method used in GPT models.\n   * Starts with characters and iteratively merges common sequences to form subwords.\n   * Efficiently handles rare words while keeping the vocabulary compact.\n\n---\n\n### **Example of token counts**\n\nLet’s take the sentence:\n\n```\nHello, world! I love programming.\n```\n\nA GPT-style tokenizer might break it into tokens like:\n\n```\n["Hello", ",", " world", "!", " I", " love", " program", "ming", "."]\n```\n\nHere, it’s **9 tokens**. Notice:\n\n* Punctuation is separate.\n* Spaces before words are included as part of the token in some tokenizers.\n* Words like “programming” are split.\n\n---\n\n### **Why tokens matter in language models**\n\n1. **Input length**\n\n   * Models have a limit on the number of tokens they can process at once (e.g., GPT-4’s 8,192 or 32k token limit).\n   * Longer text → more tokens → may exceed the model’s limit.\n\n2. **Cost & computation**\n\n   * In APIs like OpenAI’s, you pay per token.\n   * More tokens → more computation → higher cost.\n\n3. **Context & memory**\n\n   * Models can only “remember” information within a token limit.\n   * Efficient tokenization allows models to fit more meaningful text into memory.\n\n---\n\n## **2. Tokens in Programming**\n\nA token in programming is a **syntactic unit of code**:\n\n* Keywords (`if`, `for`, `return`)\n* Identifiers (`myVariable`, `functionName`)\n* Operators (`+`, `-`, `=`)\n* Literals (`42`, `"hello"`)\n* Punctuation (`;`, `{`, `}`)\n\n**Example:**\n\n```python\nx = 42 + 7\n```\n\nTokenization would produce:\n\n```\n["x", "=", "42", "+", "7"]\n```\n\nThis is similar in principle: breaking a stream of characters into meaningful chunks the compiler or interpreter can understand.\n\n---\n\n## **3. Tokens in Blockchain / Crypto**\n\nHere, a token is a **digital asset** issued on a blockchain, representing something of value:\n\n* **Utility tokens** – give access to a service (e.g., Binance Coin for exchange fees).\n* **Security tokens** – represent ownership or investment.\n* **NFTs (non-fungible tokens)** – unique digital items (art, collectibles).\n\nThis is different from NLP, but the common theme is that a token is a **unit of something meaningful**, whether text, code, or digital value.\n\n---\n\n## **Key Takeaways**\n\n* **In AI/NLP:** Token = smallest meaningful piece of text for a model to process.\n* **In programming:** Token = smallest meaningful element of code.\n* **In crypto:** Token = a digital representation of value or rights.\n\n**Analogy:** Tokens are like LEGO blocks. Each block may be a full piece (word) or a part of a bigger structure (subword). You can build complex things (sentences, programs, ecosystems) from these basic building blocks.\n\n---\n\nIf you want, I can make a **visual diagram showing how a sentence gets broken into tokens and why that matters for GPT**, which makes this concept much easier to grasp.\n\nDo you want me to do that?\n'

In [15]:
for model in client.list().models:
    print(f"{model.model},{model.details.quantization_level},{model.size/10**9:.3f}")

granite4:small-h,Q4_K_M,19.477
granite4:tiny-h,Q4_K_M,4.231
granite4:micro-h,Q4_K_M,1.943
qwen3.5:9b,Q4_K_M,6.594
lfm2.5-thinking:1.2b,Q4_K_M,0.731
qwen3.5:2b,Q8_0,2.741
qwen3.5:0.8b,Q8_0,1.036
glm-ocr:q8_0,Q8_0,1.588
gemma4:31b,Q4_K_M,19.869
gemma4:26b,Q4_K_M,17.988
gemma4:e4b,Q4_K_M,9.608
gemma4:e2b,Q4_K_M,7.162
qwen3.5:4b,Q4_K_M,3.390
embeddinggemma:300m,BF16,0.622
devstral-small-2:24b,Q4_K_M,15.177
qwen3.5:35b,Q4_K_M,23.869
gemma3:27b,Q4_K_M,17.397
qwen3.5:27b-130k,Q4_K_M,17.420
qwen3.5:27b,Q4_K_M,17.420
qwen3.5:35b-190k,Q4_K_M,23.869
qwen3.5:27b-q4_K_M,Q4_K_M,17.420
qwen3-coder-next:latest,Q4_K_M,51.742
qwen3-vl:30b,Q4_K_M,19.595
qwen3:32b,Q4_K_M,20.201
qwen3:30b,Q4_K_M,18.557
nemotron-3-nano:30b,Q4_K_M,24.272
qwen3-coder:30b,Q4_K_M,18.557
qwen3-vl:32b,Q4_K_M,20.910
glm-4.7-flash:q4_K_M,Q4_K_M,18.766
devstral-small-2:24b-100k,Q4_K_M,15.177
ministral-3:14b,Q4_K_M,9.083
qwen3-vl:8b,Q4_K_M,6.140
ministral-3:8b,Q4_K_M,6.022
qwen3-vl:4b,Q4_K_M,3.296
ministral-3:3b,Q4_K_M,2.954
qwen3-vl

In [16]:
models=[
("gemma4:31b",ctx256k),
("gemma4:26b",ctx256k),
("gemma4:e4b",ctx128k),
("gemma4:e2b",ctx128k),
("qwen3.5:27b",ctx256k),
("qwen3.5:35b",ctx256k),
("qwen3.5:9b",ctx256k),
("qwen3.5:4b",ctx256k),
("qwen3.5:2b",ctx256k),
("qwen3.5:0.8b",ctx256k),
("glm-4.7-flash:q4_K_M",ctx192k),
("gpt-oss:20b",ctx128k),
("devstral-small-2:24b",ctx384k),
("mistral-small3.2:24b",ctx128k),
("ministral-3:14b",ctx256k),
("ministral-3:8b",ctx256k),
("ministral-3:3b",ctx256k),
("lfm2.5-thinking:1.2b",ctx128k),
("granite4:small-h",ctx128k),
("granite4:tiny-h",ctx128k),
("granite4:micro-h",ctx128k),
("gemma3:27b",ctx128k),
("gemma3:12b",ctx128k),
("gemma3:4b",ctx128k),
("gemma3:1b",ctx32k),
("gemma3n:e4b",ctx32k),
("gemma3n:e2b",ctx32k),
("embeddinggemma:300m",ctx2k),
("qwen3-embedding:0.6b",ctx32k),
("nomic-embed-text:latest",ctx2k),
("glm-ocr:q8_0",ctx128k),
]

In [87]:
for model,max_context_length in models[:4]:
    context_vram_list = []
    last_context_vram = 0
    for ctx_index in range(7,35):
        context_length = context_sizes[ctx_index]
        if context_length <= max_context_length and last_context_vram < 32:
            model_vram_gb, load_time_sec = load_model()
            print(f"Model {model} loaded in {load_time_sec:.2f} sec - {model_vram_gb:.3f}GB VRAM at {int(context_length/1024)}k context length")
            context_vram_list.append(f"{model_vram_gb:.3f}")
            last_context_vram = model_vram_gb
            unload_model()
        else:
            context_vram_list.append("")
    print(f",{load_time_sec:.2f},{max_context_length},{','.join(context_vram_list)}")

Model gemma4:31b loaded in 20.51 sec - 22.302GB VRAM at 8k context length
Model gemma4:31b loaded in 8.16 sec - 22.544GB VRAM at 16k context length
Model gemma4:31b loaded in 6.61 sec - 22.837GB VRAM at 24k context length
Model gemma4:31b loaded in 6.63 sec - 23.095GB VRAM at 32k context length
Model gemma4:31b loaded in 7.48 sec - 23.397GB VRAM at 40k context length
Model gemma4:31b loaded in 7.09 sec - 23.634GB VRAM at 48k context length
Model gemma4:31b loaded in 6.81 sec - 23.928GB VRAM at 56k context length
Model gemma4:31b loaded in 7.14 sec - 24.179GB VRAM at 64k context length
Model gemma4:31b loaded in 7.24 sec - 24.473GB VRAM at 72k context length
Model gemma4:31b loaded in 7.27 sec - 24.727GB VRAM at 80k context length
Model gemma4:31b loaded in 7.28 sec - 25.029GB VRAM at 88k context length
Model gemma4:31b loaded in 7.32 sec - 25.289GB VRAM at 96k context length
Model gemma4:31b loaded in 7.40 sec - 25.591GB VRAM at 104k context length
Model gemma4:31b loaded in 7.21 sec -

In [92]:
# Constant for the date
TEST_DATE = "20260403"

# Load all dataframes once
def load_performance_data():
    dataframes = {}
    for q_level in ['fp16', 'q8_0', 'q4_0']:
        filename = f"/home/workspace/wordslab-notebooks-tutorials/perf-tests/ollama_0.10.4_context_{q_level}_{TEST_DATE}.csv"
        if os.path.exists(filename):
            dataframes[q_level] = pd.read_csv(filename)
    return dataframes

# Load dataframes once at module level
PERF_DFS = load_performance_data()

def get_max_context_length(model):
    """
    Determine the maximum context length for a specific model based on available VRAM.

    Args:
        model_name (str): Name of the model to check

    Returns:
        dict: Dictionary with quantization levels as keys and max context lengths as values
    """
    # Get current VRAM usage
    mem_info = nvmlDeviceGetMemoryInfo(handle)
    available_vram_gb = (mem_info.total - mem_info.used) / (1024**3)

    # Context column names and their corresponding lengths
    context_columns = ['8k', '16k', '24k', '32k', '40k', '48k', '56k',
                     '64k', '72k', '80k', '88k', '96k', '104k', '112k',
                     '120k', '128k', '144k', '160k', '176k', '192k',
                     '208k', '224k', '240k', '256k', '288k', '320k', '352k', '384k']

    # Parse column names to get numeric values (8, 16, 24, etc.)
    context_lengths = [int(col[:-1]) * 1024 for col in context_columns]
    
    results = {}

    for q_level, df in PERF_DFS.items():
        
        # Find the row for this model
        model_row = df[df['model'] == model]
        if model_row.empty:
            return None
    
        # Check from largest to smallest context length
        results[q_level] = None
        for i in range(len(context_columns)-1, -1, -1):
            col = context_columns[i]
            vram_used = model_row[col].item()
            
            # Skip empty values
            if pd.isna(vram_used) or vram_used == "":
                continue
            
            # Check if this context length fits in available VRAM
            if vram_used <= available_vram_gb:
                results[q_level] = context_lengths[i]
                break
                
    return results

# Example usage:
max_contexts = get_max_context_length("qwen3.5:27b")
print("qwen3.5:27b", max_contexts)
max_contexts = get_max_context_length("qwen3.5:35b")
print("qwen3.5:35b", max_contexts)
max_contexts = get_max_context_length("gemma4:31b")
print("gemma4:31b", max_contexts)
max_contexts = get_max_context_length("gemma4:26b")
print("gemma4:26b", max_contexts)

qwen3.5:27b {'fp16': 90112, 'q8_0': 131072, 'q4_0': 212992}
qwen3.5:35b {'fp16': 131072, 'q8_0': 163840, 'q4_0': 212992}
gemma4:31b {'fp16': 65536, 'q8_0': 131072, 'q4_0': 245760}
gemma4:26b {'fp16': 262144, 'q8_0': 262144, 'q4_0': 262144}


In [93]:
ollama_version=!ollama --version
ollama_version=ollama_version[0].split()[-1]
ollama_version

'0.20.4-rc1'

> OLLAMA_HOST=0.0.0.0 OLLAMA_KEEP_ALIVE=-1 OLLAMA_FLASH_ATTENTION=1 OLLAMA_NEW_ENGINE=1 OLLAMA_MULTIUSER_CACHE=1 ollama serve &

https://github.com/ggml-org/llama.cpp/issues/21385

Finding 1: Lossless q4_0 on Hybrid Models

Why Hybrid Models Tolerate q4_0
Qwen3.5 uses only 8 of 32 layers for full attention with KV cache. The other 24 layers use linear attention (Gated Delta Net) with no KV cache. Similarly, Gemma 4 uses 10 of 60 layers for global attention.

The linear/sliding attention layers act as error correction -- quantization noise in the few attention layers is absorbed by the surrounding layers. Standard models (Llama, Mistral) have no such correction because all layers use full attention.

In [98]:
for model,_ in models[:4]:    
    # Get the maximum available context length
    context_length = get_max_context_length(model)["fp16"]
    print(model,get_max_context_length(model), get_max_context_length(model)["fp16"])
    
    # Load the mode and tokenize the test text
    base_tokens = compute_prompt_tokens(base_text)   

    # Test at all prompt sizes until context_length - 100 (output tokens)
    previous_prompt_tokens = -1
    for prompt_size in context_sizes:
        prompt_repeat = int(prompt_size/base_tokens)
        prompt_tokens = prompt_repeat * base_tokens
        if prompt_tokens > (context_length - 100):
            break
        if prompt_tokens != previous_prompt_tokens:
            previous_prompt_tokens = prompt_tokens
            start_time = time.time()
            real_prompt_tokens,prompt_tokens_per_sec, output_tokens,output_tokens_per_sec = measure_perfs(prompt_repeat)
            end_time = time.time()
            elapsed_time = end_time - start_time
            print(f"{gpu_model},{model},{real_prompt_tokens},{int(prompt_tokens_per_sec)},{output_tokens_per_sec:.2f}")
            if elapsed_time > 60:
                break

    unload_model()

gemma4:31b {'fp16': 73728, 'q8_0': 131072, 'q4_0': 245760} 73728
RTX 5090 32GB,gemma4:31b,15,136,63.53
RTX 5090 32GB,gemma4:31b,1273,3468,60.57
RTX 5090 32GB,gemma4:31b,2532,6490,60.24
RTX 5090 32GB,gemma4:31b,3791,9682,60.26
RTX 5090 32GB,gemma4:31b,5050,11897,54.44
RTX 5090 32GB,gemma4:31b,6309,14176,59.88
RTX 5090 32GB,gemma4:31b,7568,17925,59.79
RTX 5090 32GB,gemma4:31b,15122,5653,53.93
RTX 5090 32GB,gemma4:31b,23935,6855,52.67
RTX 5090 32GB,gemma4:31b,31489,9313,52.32
RTX 5090 32GB,gemma4:31b,40302,9087,51.53
RTX 5090 32GB,gemma4:31b,47856,11050,50.45
RTX 5090 32GB,gemma4:31b,56669,10158,49.15
RTX 5090 32GB,gemma4:31b,64223,12061,48.44
RTX 5090 32GB,gemma4:31b,71777,12509,46.72
gemma4:26b {'fp16': 262144, 'q8_0': 262144, 'q4_0': 262144} 262144
RTX 5090 32GB,gemma4:26b,15,98,171.68
RTX 5090 32GB,gemma4:26b,1273,7674,165.97
RTX 5090 32GB,gemma4:26b,2532,14462,166.65
RTX 5090 32GB,gemma4:26b,3791,20587,161.54
RTX 5090 32GB,gemma4:26b,5050,27142,148.93
RTX 5090 32GB,gemma4:26b,6309,31

> default

RTX 5090 32GB,qwen3.5:27b,1276,1429,61.59
RTX 5090 32GB,qwen3.5:27b,2538,1860,61.08
RTX 5090 32GB,qwen3.5:27b,3800,1779,60.61
RTX 5090 32GB,qwen3.5:27b,5062,3036,60.51
RTX 5090 32GB,qwen3.5:27b,6324,5487,60.12
RTX 5090 32GB,qwen3.5:27b,7586,3944,60.19
RTX 5090 32GB,qwen3.5:27b,8848,5898,59.56
RTX 5090 32GB,qwen3.5:27b,16420,3168,57.00
RTX 5090 32GB,qwen3.5:27b,25254,3530,56.12

RTX 5090 32GB,gemma4:31b,1278,593,13.89
RTX 5090 32GB,gemma4:31b,2537,464,10.32
RTX 5090 32GB,gemma4:31b,3796,271,8.20
RTX 5090 32GB,gemma4:31b,5055,170,6.64

RTX 5090 32GB,qwen3.5:35b,1276,1937,146.41
RTX 5090 32GB,qwen3.5:35b,2538,3601,146.45
RTX 5090 32GB,qwen3.5:35b,3800,3083,145.31
RTX 5090 32GB,qwen3.5:35b,5062,5567,144.54
RTX 5090 32GB,qwen3.5:35b,6324,9936,142.78
RTX 5090 32GB,qwen3.5:35b,7586,6877,142.07
RTX 5090 32GB,qwen3.5:35b,8848,10011,135.18
RTX 5090 32GB,qwen3.5:35b,16420,5576,129.49
RTX 5090 32GB,qwen3.5:35b,25254,6394,129.98

> new engine, multi user cache, default kv cache fp16

RTX 5090 32GB,qwen3.5:27b,1276,1571,61.51
RTX 5090 32GB,qwen3.5:27b,2538,1940,61.16
RTX 5090 32GB,qwen3.5:27b,3800,1781,61.13
RTX 5090 32GB,qwen3.5:27b,5062,3095,60.81
RTX 5090 32GB,qwen3.5:27b,6324,5677,60.51
RTX 5090 32GB,qwen3.5:27b,7586,3843,60.28
RTX 5090 32GB,qwen3.5:27b,8848,5937,59.69
RTX 5090 32GB,qwen3.5:27b,16420,3182,57.25
RTX 5090 32GB,qwen3.5:27b,25254,3611,56.10

RTX 5090 32GB,gemma4:31b,1278,636,13.70
RTX 5090 32GB,gemma4:31b,2537,473,10.65
RTX 5090 32GB,gemma4:31b,3796,259,8.41

RTX 5090 32GB,gemma4:26b,1278,2022,17.76
RTX 5090 32GB,gemma4:26b,2537,1692,17.75
RTX 5090 32GB,gemma4:26b,3796,1615,15.93

RTX 5090 32GB,qwen3.5:35b,1276,2489,147.22
RTX 5090 32GB,qwen3.5:35b,2538,3849,144.65
RTX 5090 32GB,qwen3.5:35b,3800,3338,144.32
RTX 5090 32GB,qwen3.5:35b,5062,5615,143.78
RTX 5090 32GB,qwen3.5:35b,6324,10141,143.09
RTX 5090 32GB,qwen3.5:35b,7586,7179,142.75
RTX 5090 32GB,qwen3.5:35b,8848,10602,134.36
RTX 5090 32GB,qwen3.5:35b,16420,5599,128.69
RTX 5090 32GB,qwen3.5:35b,25254,6366,130.19

> new engine, multi user cache, kv cache q8_0

RTX 5090 32GB,qwen3.5:27b,1276,740,60.23
RTX 5090 32GB,qwen3.5:27b,2538,948,59.81
RTX 5090 32GB,qwen3.5:27b,3800,937,60.93
RTX 5090 32GB,qwen3.5:27b,5062,1605,59.66
RTX 5090 32GB,qwen3.5:27b,6324,2907,59.18
RTX 5090 32GB,qwen3.5:27b,7586,2157,58.76
RTX 5090 32GB,qwen3.5:27b,8848,3293,58.84
RTX 5090 32GB,qwen3.5:27b,16420,1696,56.89
RTX 5090 32GB,qwen3.5:27b,25254,1888,56.96

RTX 5090 32GB,qwen3.5:35b,1276,2539,144.19
RTX 5090 32GB,qwen3.5:35b,2538,3444,143.58
RTX 5090 32GB,qwen3.5:35b,3800,3039,143.19
RTX 5090 32GB,qwen3.5:35b,5062,5096,144.39
RTX 5090 32GB,qwen3.5:35b,6324,9564,143.91
RTX 5090 32GB,qwen3.5:35b,7586,7119,142.24
RTX 5090 32GB,qwen3.5:35b,8848,9844,143.90
RTX 5090 32GB,qwen3.5:35b,16420,5648,136.08
RTX 5090 32GB,qwen3.5:35b,25254,6174,134.39

> new engine, multi user cache, kv cache q4_0

RTX 5090 32GB,qwen3.5:27b,1276,1588,61.20
RTX 5090 32GB,qwen3.5:27b,2538,1897,60.87
RTX 5090 32GB,qwen3.5:27b,3800,1748,60.75
RTX 5090 32GB,qwen3.5:27b,5062,3104,60.62
RTX 5090 32GB,qwen3.5:27b,6324,5561,60.18
RTX 5090 32GB,qwen3.5:27b,7586,3904,59.74
RTX 5090 32GB,qwen3.5:27b,8848,6137,60.34
RTX 5090 32GB,qwen3.5:27b,16420,3098,57.44
RTX 5090 32GB,qwen3.5:27b,25254,3543,56.46

RTX 5090 32GB,gemma4:26b,1278,1417,24.36
RTX 5090 32GB,gemma4:26b,2537,1217,22.71
RTX 5090 32GB,gemma4:26b,3796,1132,24.08
RTX 5090 32GB,gemma4:26b,5055,1119,21.71

RTX 5090 32GB,qwen3.5:35b,1276,2068,145.99
RTX 5090 32GB,qwen3.5:35b,2538,3642,144.64
RTX 5090 32GB,qwen3.5:35b,3800,3139,144.99
RTX 5090 32GB,qwen3.5:35b,5062,6023,144.43
RTX 5090 32GB,qwen3.5:35b,6324,10432,144.15
RTX 5090 32GB,qwen3.5:35b,7586,7614,143.12
RTX 5090 32GB,qwen3.5:35b,8848,11144,143.37
RTX 5090 32GB,qwen3.5:35b,16420,5760,135.58
RTX 5090 32GB,qwen3.5:35b,25254,6406,134.26